<a href="https://colab.research.google.com/github/BrianCapobianco22/ml-algorithms-from-scratchANDprojects/blob/main/My_GPT_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.__version__)       ## seing if torch is working on our computer
print(torch.cuda.is_available())            ## were seing if we can use the NVIDIA GPU (returns a boolean)

device= "cuda" if torch.cuda.is_available() else "cpu"            ## if the GPU is avalible we will use it if not we will use a cpu
print(device)

2.14.1+cu130
True
cuda


In [2]:
import urllib.request

url= "https://www.gutenberg.org/cache/epub/11/pg11.txt"  ## downloading the text
urllib.request.urlretrieve(url,"input.txt")   ## url is telling python where to retrive the text and "input.txt" is telling python where to store it
with open("input.txt" ,  "r", encoding="utf-8-sig") as f: ## were telling python to open the text and to read it and were going to save it as f
  text=f.read()
  print(len(text))
  print(text[:500])    ## displaying the first 500 characters



163950
The Project Gutenberg eBook of Alice's Adventures in Wonderland
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using 


In [3]:
chars= sorted(list(set(text)))   ## we are going to get unique values in the text so we can assign IDs to them so the tokenizers can work. a set only keeps unique IDS and we are going to sort it too
vocab_size = len(text)

print(chars)
print(vocab_size)

['\n', ' ', '!', '#', '$', '%', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'ù', '—', '‘', '’', '“', '”', '•', '™']
163950


In [4]:
## tokenizer
string_to_int= {}

for i, ch in enumerate(chars):    ## input text to numbers, these are just dictionarys so we can store key vale pairs we will use these dictionaries when we create our actual encode and decode functions
  string_to_int[ch]=i


int_to_string={}

for i, ch in enumerate(chars):    ## output numbers to text
  int_to_string[i]=ch


def encode(s):
  return_int_list=[]
  for character in s:
    return_int_list.append(string_to_int[character])
  return return_int_list


def decode(tokens):
  token_list = []
  for token in tokens:
    token_list.append(int_to_string[token])
  return ''.join(token_list)

encoded_text=encode(text)

decoded_text=decode(encoded_text)






In [5]:

import torch

data= torch.tensor(encoded_text, dtype=torch.long)

n = int(0.9*len(data))

training_data=data[:n]

testing_data=data[n:]

print(len(training_data))

print(len(testing_data))


147555
16395


In [6]:
block_size=8

x=training_data[:block_size]  ## all 8 IDS
y=training_data[1:block_size+1]  ## starting at the second character.

print(x)
print(y)

tensor([47, 64, 61,  1, 43, 74, 71, 66])
tensor([64, 61,  1, 43, 74, 71, 66, 61])


In [16]:
batch_size=4 ## how many training examples we want GPT to look at all at the same time
block_size= 8   ## number of characters we want the GPT to look at ata a time

def get_batch():
  ix=torch.randint(0,len(training_data)- block_size, (batch_size,))


  x_list=[]
  y_list=[]

  for i in ix:
    x=training_data[i:i+block_size]
    y=training_data[i+1:i+block_size+1]
    x_list.append(x)
    y_list.append(y)

  x=torch.stack(x_list)
  y=torch.stack(y_list)
  return x,y
    ## in this function we are randomly going throughout the training data to select 4 strating postions and then we will loop through each starting postion to get 8 ids for each starting postion
## x will contain all 8 IDS starting at postion 0
## y will contain all 8 IDS starting at postion 1




In [17]:
## embedding layer

import torch.nn as NN

vocab_size= len(chars)
embedding_size_for_each_ID=32

token_embedding_table=NN.Embedding(vocab_size,embedding_size_for_each_ID)


x , y =get_batch()

embeddings=token_embedding_table(x)

print(x.shape)
print(embeddings.shape)

torch.Size([4, 8])
torch.Size([4, 8, 32])


In [23]:
## prediction layer
import torch.nn as NN


prediction_layer=NN.Linear(32, vocab_size)

logits= prediction_layer(embeddings)

print(logits.shape)

predicted_ids = torch.argmax(logits, dim=-1)

print(predicted_ids)
for i in range(batch_size):
    print("Input:     ", decode(x[i].tolist()))
    print("Predicted: ", decode(predicted_ids[i].tolist()))
    print("Actual:    ", decode(y[i].tolist()))
    print()

torch.Size([4, 8, 91])
tensor([[17, 83, 84,  0, 20, 25, 25, 10],
        [17, 75, 20, 20, 20, 17, 17, 36],
        [78, 83, 25, 36,  0, 17, 75, 83],
        [ 0, 17, 87, 78, 17, 33, 20, 25]])
Input:       interru
Predicted:  2ù—
5::+
Actual:     interrup

Input:       well as
Predicted:  2s55522I
Actual:     well as 

Input:      first wi
Predicted:  vù:I
2sù
Actual:     irst wit

Input:      t of her
Predicted:  
2“v2F5:
Actual:      of her




In [ ]:
##calculating loss
